# Interactive energy-model organoid viewer

Inspect one saved validation organoid at a time with three rotatable surface-mesh panels: **true mean curvature**, **predicted mean curvature**, and **exclusive lineage identity**. The mesh projection, diverging curvature colors and marker palette follow `organoid_predictions.ipynb`. Both curvature panels share identical color limits; rotating or zooming one panel updates the other two.

The default is the **measured-mean/SD normalized, all-to-all one-hop presence energy model**, which performed best among the models in the conditional-allocation experiment. Each fold starts at the accommodation value selected on its **inner training holdout**, not the best outer-validation score. This is γ=0.8 for fold 0 and γ=0.4 for folds 1–4. Change MODEL_FOLD near the top to view another fold, or use the widget to explore other γ values.

Curvature is displayed in physical units. For the conditional model, prediction is **measured organoid mean + saved effective SD × predicted standardized field**. Thus the organoid supplies its measured mean and spread; this is conditional reconstruction, not fate-only prediction of morphology. Historical energy runs remain supported and instead restore their fitted global baseline. “True” means the target after the original training-fitted outlier cleanup, not untouched mesh-vertex curvature. Display normalization affects mesh coordinates only.

Saved evaluation predictions are used when available; otherwise only the selected organoid is inferred. Nothing is trained and no data are resampled. Ordinary organoids are held out from the selected fold; spherical organoids are external to every training fold. Exclusive identities include Unassigned; unmapped mesh vertices are separate. Mesh files must remain accessible. Run all cells and click **Display organoid**. Plotly is embedded locally; the notebook must be trusted and support ipywidgets.

In [1]:
from pathlib import Path
import sys
import json
import html
from functools import lru_cache
import numpy as np
import pandas as pd
import torch
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src/models/gnn.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.artifacts.bundle import load_bundle
from src.artifacts.runs import AnalysisRun
from src.artifacts.paths import resolve_training_run
from src.training.coupled_fit import graph_samples
from src.data.neighborhood_counts import fate_identities
from src.plotting.mesh_plots import project_node_quantities_to_mesh, plot_projected_true_vs_pred, plot_categorical_mesh
from src.plotting.motif_plots import DEFAULT_MARKER_COLORS

## Settings
Select a run and one fold. Leave INITIAL_GAMMA=None to use that fold's saved inner-selected accommodation. The conditional default uses standardized targets; TARGET_MODE='mean_only' selects the control. Cohort and gamma can be changed in the widget without reloading. CURVATURE_LIMITS=None gives a shared symmetric color scale for the chosen organoid.

In [2]:
# Saved run and model
TRAINING_RUN = 'training_results/shape_conditioned_energy_training/observed_mean_std_presence_20261002_144511'  # Best conditional-allocation model family.
MODEL_FOLD = 0  # One saved fold; ordinary organoids shown are held out from this fold.
TARGET_MODE = 'standardized'  # Conditional runs: 'standardized' (mean+SD) or 'mean_only'; historical runs ignore this.
EVALUATION_TAG = None  # None chooses the matching evaluation folder; missing archives trigger inference.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Used only when cached predictions are absent.

# Initial browser selection
INITIAL_GAMMA = None  # None uses this fold's inner-selected gamma; a number manually overrides it.
INITIAL_COHORT = 'val'  # 'val': ordinary validation; 'spherical': separate spherical validation.
INITIAL_ORGANOID = None  # Saved organoid ID; None starts with the first displayed candidate.
ORDER_BY = 'mse'  # 'mse': largest error first; 'N': largest first; 'name': alphabetical.

# Mesh appearance
CURVATURE_LIMITS = None  # Optional (minimum, maximum) shared by truth and prediction; None: symmetric automatic limits.
COLORSCALE = 'RdBu_r'  # Match the existing curvature visualization palette.
FIGURE_HEIGHT = 620  # Height of the three-panel display in pixels.
VIEW = dict(azim=-135, elev=25, projection='orthographic')  # Shared initial mesh view.
LINK_CAMERAS = True  # Synchronize rotation and zoom across the three panels.

## Restore the cohort, fold and model catalog
Select the presence-energy family and target normalization before reading scores, so models with the same gamma cannot be mixed. Restore exact validation membership and saved moments. Original mesh metadata come from the conditional run's recorded cohort reference; no model or baseline is refitted. The default gamma comes from inner validation, and per-organoid ranking uses the selected model's physical MSE.

In [3]:
run=AnalysisRun(resolve_training_run(PROJECT_ROOT,'shape_conditioned_energy_training',TRAINING_RUN))
if not (run.directory/'complete.json').exists():raise ValueError('Select a completed training run.')
conditional='target_mode' in run.records.columns
records=run.records[(run.records.family=='energy')&(run.records.fold==MODEL_FOLD)].copy()
if conditional:
    if TARGET_MODE not in ['standardized','mean_only']:raise ValueError('Choose standardized or mean_only target mode.')
    records=records[(records.target_mode==TARGET_MODE)&(records.variant=='presence_energy')].copy()
if records.empty or records.gamma.duplicated().any():raise ValueError('Select a fold with one energy checkpoint per gamma in this model family.')
records=records.sort_values('gamma');record_by_gamma={float(r.gamma):r for r in records.itertuples()}
if conditional:
    selection=pd.read_csv(run.directory/'selected_models.csv')
    chosen=selection[(selection.fold==MODEL_FOLD)&(selection.target_mode==TARGET_MODE)]
    if len(chosen)!=1 or chosen.iloc[0].key not in set(records.key):raise ValueError('Missing or ambiguous saved inner selection.')
    selected_gamma=float(chosen.iloc[0].gamma)
    bundles=records.input_bundle.unique()
    if len(bundles)!=1:raise ValueError('Ambiguous fold input bundle.')
    data=load_bundle(run.directory/bundles[0])
    source=Path(run.settings['reference_run'])
    if not source.is_absolute():source=PROJECT_ROOT/source
    raw_graphs=load_bundle(source/'cohort')['raw_graphs']
    groups=data['physical_groups']
    prediction_context='Conditional reconstruction: measured organoid mean + effective SD × standardized prediction' if TARGET_MODE=='standardized' else 'Conditional reconstruction: measured organoid mean + predicted residual'
else:
    data=run.fold_inputs(MODEL_FOLD);raw_graphs=data['raw_graphs'];groups=data['groups']
    inner_scores=[(load_bundle(run.directory/r.bundle)['metrics']['best']['inner_mse'],float(r.gamma)) for r in records.itertuples()]
    selected_gamma=min(inner_scores)[1]
    prediction_context='Physical prediction: fitted global baseline + predicted residual'
initial_gamma=selected_gamma if INITIAL_GAMMA is None else float(INITIAL_GAMMA)
if initial_gamma not in record_by_gamma:raise ValueError(f'Available gamma values: {list(record_by_gamma)}')
evaluation_tag=EVALUATION_TAG or ('conditional_allocation' if conditional else 'accommodation_profiles')
marker_names=data['marker_names'];identity_names=list(marker_names)+['Unassigned']
if run.settings['target_indices']!=[1]:raise ValueError('This viewer displays scalar mean-curvature energy models.')
graphs={role:{str(g.organoid_str):g for g in groups.get(role,[])} for role in ['val','spherical']}
split=next(s for s in json.loads((run.directory/'splits.json').read_text()) if s['fold']==MODEL_FOLD)
if list(graphs['val'])!=split['validation'] or list(graphs['spherical'])!=split['spherical_validation']:raise ValueError('Validation membership mismatch.')
if set(graphs['val'])&set(split['train']) or set(graphs['spherical'])&set(split['train']):raise ValueError('Training cells cannot appear in the validation browser.')
if INITIAL_COHORT not in graphs or not graphs[INITIAL_COHORT]:raise ValueError('The initial validation cohort is empty or invalid.')
if INITIAL_ORGANOID is not None and INITIAL_ORGANOID not in graphs[INITIAL_COHORT]:raise ValueError('Initial organoid is not in the selected cohort/fold.')
cohort_info=pd.read_csv(run.directory/'cohort.csv').set_index('organoid_str')
saved_scores=pd.read_csv(run.directory/'validation_mse.csv')
saved_scores=saved_scores[(saved_scores.fold==MODEL_FOLD)&saved_scores.key.isin(records.key)].copy()
if saved_scores.duplicated(['gamma','cohort','organoid_str']).any():raise ValueError('Duplicate scores after selecting the model family.')
marker_colors={name:DEFAULT_MARKER_COLORS.get(name,'#EBEBEB') for name in identity_names}
marker_colors['Unassigned']=DEFAULT_MARKER_COLORS.get('none','#EBEBEB')
if ORDER_BY not in ['mse','N','name']:raise ValueError('ORDER_BY must be mse, N or name.')
if CURVATURE_LIMITS is not None:
    if len(CURVATURE_LIMITS)!=2 or not np.isfinite(CURVATURE_LIMITS).all() or CURVATURE_LIMITS[0]>=CURVATURE_LIMITS[1]:raise ValueError('CURVATURE_LIMITS must be two finite increasing values.')
print(f'{run.directory.name} | fold {MODEL_FOLD} | inner-selected gamma={selected_gamma:g}')
print(prediction_context)
print(f'{len(graphs["val"])} ordinary and {len(graphs["spherical"])} spherical validation organoids')

observed_mean_std_presence_20261002_144511 | fold 0 | inner-selected gamma=0.8
Conditional reconstruction: measured organoid mean + effective SD × standardized prediction
174 ordinary and 324 spherical validation organoids


## Single-organoid prediction and mesh rendering
Validate cached physical targets against the selected fold. If an archive is absent, infer only that organoid, without passing its measured node targets to the model. Reconstruct with its saved mean and effective SD for the conditional model, or its fitted baseline for historical models. Cache at most two archives/checkpoints.

Project all panels through the same saved node-to-mesh ownership. Curvature panels share a color scale. The linked-camera display replaces the old figure on every click, avoiding duplicate figures or metadata rows.

In [4]:
@lru_cache(maxsize=2)
def prediction_archive(key,role):
    path=run.directory/'analysis'/evaluation_tag/'predictions'/f'{key}_{role}.npz'
    if not path.exists():return None
    with np.load(path,allow_pickle=False) as archive:
        return {name:archive[name].copy() for name in ['truth','prediction','offsets','organoid_ids']}

@lru_cache(maxsize=2)
def energy_model(key):
    record=records[records.key==key].iloc[0]
    model=load_bundle(run.directory/record.bundle)['model']
    if model.fixed_strength!=float(record.gamma):raise ValueError('Checkpoint gamma does not match catalog.')
    if conditional and not model.zero_mean_output:raise ValueError('Conditional checkpoint lacks the fitted zero-mean projection.')
    return model

def organoid_prediction(oid,role,gamma):
    graph=graphs[role][oid];record=record_by_gamma[float(gamma)]
    if conditional:
        truth=graph.y.detach().cpu().numpy().reshape(-1)
        moments=data['moments'][oid]
        offset=moments.mean;scale=moments.scale if TARGET_MODE=='standardized' else 1.
    else:
        offset=np.asarray(data['baseline_offsets'][oid]).reshape(-1);scale=1.
        truth=np.asarray(data['transform'].inverse(graph.y.detach().cpu().numpy())).reshape(-1)+offset
    archive=prediction_archive(record.key,role)
    if archive is not None:
        where=np.flatnonzero(archive['organoid_ids']==oid)
        if len(where)!=1:raise ValueError('Prediction archive must contain the selected organoid exactly once.')
        k=int(where[0]);start,stop=archive['offsets'][k:k+2]
        prediction=archive['prediction'][start:stop].copy();cached_truth=archive['truth'][start:stop]
        if cached_truth.shape!=truth.shape or not np.allclose(cached_truth,truth,rtol=1e-6,atol=1e-9):raise ValueError("Saved prediction archive disagrees with the selected fold physical targets.")
    else:
        model=energy_model(record.key)
        samples=graph_samples([graph],2)
        samples[0].pop('y')  # Inference cannot use the measured node targets.
        prediction=offset+scale*model.predict_samples(samples,device=DEVICE,rtol=run.settings['solver_rtol'])[0]
    if prediction.shape!=truth.shape or not np.isfinite(prediction).all():raise ValueError('Invalid prediction shape or nonfinite values.')
    return truth,prediction,graph

def make_organoid_figure(oid,role,gamma):
    truth,prediction,prepared=organoid_prediction(oid,role,gamma)
    identity=fate_identities(prepared.x[:,:len(marker_names)])
    graph=raw_graphs[oid].clone()
    if len(graph.x)!=len(identity):raise ValueError('Raw mesh graph and prepared nodes differ.')
    projection=project_node_quantities_to_mesh(graph,node_true=truth,node_pred=prediction)
    owner=np.asarray(projection['vertex_owner']);valid=owner>=0
    labels=np.full(len(owner),'No mapped cell',dtype=object)
    labels[valid]=np.asarray(identity_names,dtype=object)[identity[owner[valid]]]
    scalar=plot_projected_true_vs_pred(projection,colorscale=COLORSCALE,view=VIEW)
    categorical=plot_categorical_mesh(projection['mesh'],labels,category_order=identity_names,category_colors=marker_colors,
        baseline_name='No mapped cell',baseline_color='#BDBDBD',view=VIEW)
    fig=make_subplots(rows=1,cols=3,specs=[[{'type':'scene'}]*3],subplot_titles=['True mean curvature','Predicted mean curvature','Exclusive lineage identities'],horizontal_spacing=.035)
    for trace in scalar.data:
        column=1 if trace.scene in [None,'scene'] else 2
        trace.showlegend=False
        if CURVATURE_LIMITS is not None:trace.update(cmin=CURVATURE_LIMITS[0],cmax=CURVATURE_LIMITS[1])
        trace.update(showscale=column==2,colorbar=dict(title='Mean H',x=.65,len=.72,thickness=12),customdata=np.column_stack([owner,labels]),hovertemplate='Cell %{customdata[0]}<br>%{customdata[1]}<br>H=%{intensity:.5g}<extra></extra>')
        fig.add_trace(trace,row=1,col=column)
    for trace in categorical.data:fig.add_trace(trace,row=1,col=3)
    scene=scalar.layout.scene.to_plotly_json();scene.pop('domain',None)
    scene.setdefault('camera',{})['projection']={'type':VIEW.get('projection','orthographic')}
    for name in ['scene','scene2','scene3']:fig.update_layout(**{name:scene})
    mse=float(np.mean((prediction-truth)**2))
    fig.update_layout(height=FIGURE_HEIGHT,autosize=True,title=f'{oid} | fold {MODEL_FOLD} | gamma={float(gamma):g} | '+('measured mean+SD' if conditional and TARGET_MODE=='standardized' else ('measured mean' if conditional else 'baseline restored')),
        margin=dict(l=0,r=15,t=65,b=70),legend=dict(orientation='h',x=.5,xanchor='center',y=-.04),paper_bgcolor='white')
    return fig,mse

CAMERA_SCRIPT = r"""
const gd = document.getElementById('{plot_id}');
let syncing = false;
gd.on('plotly_relayout', function(event) {
    if (syncing) return;
    const key = Object.keys(event).find(k => /^(scene|scene2|scene3)\.camera$/.test(k));
    if (!key) return;
    const camera = event[key];
    const update = {};
    ['scene','scene2','scene3'].forEach(s => { update[s+'.camera'] = camera; });
    syncing = true;
    Plotly.relayout(gd, update).then(() => { syncing=false; }, () => { syncing=false; });
});
"""

def figure_frame(figure):
    document=figure.to_html(full_html=True,include_plotlyjs=True,config={'responsive':True,'displaylogo':False},post_script=CAMERA_SCRIPT if LINK_CAMERAS else None)
    return '<iframe title="Organoid curvature and identity panels" style="width:100%;height:'+str(FIGURE_HEIGHT+30)+'px;border:0" sandbox="allow-scripts allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>' 

## Browse one organoid
The widget starts at the inner-selected gamma, labelled in its dropdown. Select an organoid and click **Display organoid**. The persistent caption states which measured information reconstructs the prediction. The single metadata row reports cell count, sphericity, physical MSE and—on conditional runs—the measured mean/SD and effective SD.

Changing a selection clears the old preview. Ordinary organoids are validation-only for this fold; spherical organoids are external to every fold. Change MODEL_FOLD near the top and rerun to inspect another fold; leaving INITIAL_GAMMA=None automatically picks its saved selection. No models are retrained.

In [5]:
def build_organoid_browser():
    style={'description_width':'initial'}
    cohort_control=widgets.Dropdown(options=[('Ordinary validation','val'),('Spherical validation','spherical')],value=INITIAL_COHORT,description='Cohort',style=style)
    gamma_control=widgets.Dropdown(options=[(f'{g:g}'+(' (inner-selected)' if g==selected_gamma else ''),g) for g in record_by_gamma],value=initial_gamma,description='Accommodation γ',style=style)
    organoid_control=widgets.Dropdown(options=[],description='Organoid',style=style,layout=widgets.Layout(width='98%'))
    render_button=widgets.Button(description='Display organoid',button_style='primary')
    context=widgets.HTML(value='<p>'+html.escape(prediction_context)+'</p>')
    status=widgets.HTML();preview=widgets.HTML();canvas=widgets.HTML()
    state={'updating':False,'initial':True,'last_figure':None,'error':None}

    def preview_selection(change=None):
        if state['updating']:return
        canvas.value='';state['last_figure']=None;state['error']=None
        oid=organoid_control.value
        if oid is None:preview.value='';return
        row=cohort_info.loc[[oid],['timepoint','N','sphericity']].copy()
        match=saved_scores[(saved_scores.cohort==cohort_control.value)&(saved_scores.gamma==gamma_control.value)&(saved_scores.organoid_str==oid)]
        if conditional:
            m=data['moments'][oid];row['measured_mean']=m.mean;row['measured_SD']=m.std;row['effective_SD']=m.scale
        row['validation_MSE']=float(match.iloc[0].mse) if len(match)==1 else np.nan
        preview.value=row.to_html(border=0,escape=True)

    def refresh_options(change=None):
        role=cohort_control.value;prior=INITIAL_ORGANOID if state['initial'] else organoid_control.value
        candidates=cohort_info.loc[list(graphs[role])].copy()
        mse=saved_scores[(saved_scores.cohort==role)&(saved_scores.gamma==gamma_control.value)].set_index('organoid_str').mse
        candidates['mse']=mse.reindex(candidates.index)
        candidates=candidates.sort_index() if ORDER_BY=='name' else candidates.sort_values('mse' if ORDER_BY=='mse' else 'N',ascending=False)
        state['updating']=True
        try:
            organoid_control.options=[(f'{oid} | N={int(row.N)} | Q={row.sphericity:.3f}',oid) for oid,row in candidates.iterrows()]
            organoid_control.value=prior if prior in candidates.index else (candidates.index[0] if len(candidates) else None)
            render_button.disabled=organoid_control.disabled=not len(candidates)
        finally:state['updating']=False;state['initial']=False
        status.value=f'{len(candidates)} organoids | fold {MODEL_FOLD} | saved model {html.escape(record_by_gamma[float(gamma_control.value)].key)}'
        preview_selection()

    def render_selected(button=None):
        if organoid_control.value is None:return
        render_button.disabled=True;canvas.value='';state['error']=None
        try:
            figure,mse=make_organoid_figure(organoid_control.value,cohort_control.value,gamma_control.value)
            canvas.value=figure_frame(figure);state['last_figure']=figure
            status.value=f'{len(graphs[cohort_control.value])} organoids | fold {MODEL_FOLD} | gamma={gamma_control.value:g} | displayed MSE={mse:.7g}'
        except Exception as exc:
            state['error']=f'{type(exc).__name__}: {exc}'
            canvas.value='<p><b>Unable to display this organoid:</b> '+html.escape(state['error'])+'</p>'
        finally:render_button.disabled=False

    cohort_control.observe(refresh_options,names='value');gamma_control.observe(refresh_options,names='value')
    organoid_control.observe(preview_selection,names='value');render_button.on_click(render_selected)
    refresh_options()
    browser=widgets.VBox([context,widgets.HBox([cohort_control,gamma_control]),status,organoid_control,preview,render_button,canvas])
    return browser,dict(cohort=cohort_control,gamma=gamma_control,organoid=organoid_control,render=render_button,canvas=canvas,preview=preview,status=status,state=state)

if 'browser' in globals():
    for child in browser.children:child.close()
    browser.close()
browser,controls=build_organoid_browser()
display(browser)